# Microsoft Fabric Runtime 1.2 to 2.0 Upgrade Notebook

Use this notebook to prepare for and validate an upgrade from Microsoft Fabric Runtime 1.2 to Runtime 2.0. Runtime selection is controlled by the Fabric notebook or environment settings, so the notebook focuses on baseline capture, upgrade steps, and smoke tests that should be run before and after switching the runtime.

> Run the baseline cells while still on Runtime 1.2, change the notebook/environment runtime to 2.0, then rerun the validation cells.


## 1. Capture the current runtime baseline

Run this section before the upgrade and save the output with your deployment notes. It records the Python version, Spark version, Spark application id, and selected Spark configuration values that are useful when comparing Runtime 1.2 and Runtime 2.0 behavior.


In [ ]:
from datetime import datetime, timezone
import json
import platform

baseline = {
    'captured_at_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(),
    'platform': platform.platform(),
}

if 'spark' in globals():
    baseline['spark_version'] = spark.version
    baseline['spark_app_id'] = spark.sparkContext.applicationId
    config_prefixes = (
        'spark.databricks.delta',
        'spark.sql.catalog',
        'spark.sql.extensions',
        'spark.sql.legacy',
        'spark.sql.session.timeZone',
    )
    baseline['selected_spark_conf'] = {
        key: value
        for key, value in sorted(spark.sparkContext.getConf().getAll())
        if key.startswith(config_prefixes)
    }
else:
    baseline['spark'] = 'No active Spark session was found. Attach this notebook to a Fabric Spark runtime and rerun.'

print(json.dumps(baseline, indent=2))


## 2. Inventory important Python dependencies

Runtime upgrades can change preinstalled library versions. Add any business-critical packages to `packages_to_check` before you run the cell. Compare the output before and after moving to Runtime 2.0.


In [ ]:
from importlib import metadata

packages_to_check = [
    'pandas',
    'numpy',
    'pyarrow',
    'pyspark',
    'delta-spark',
]

for package in packages_to_check:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: not installed')


## 3. Switch the Fabric runtime to 2.0

Code cells cannot directly change the runtime version. Use the Fabric UI to perform the upgrade:

1. Open the notebook or the attached Fabric environment.
2. Open **Settings** / **Environment**.
3. Change the runtime from **Runtime 1.2** to **Runtime 2.0**.
4. Save the setting and restart the Spark session.
5. Rerun the baseline and validation cells in this notebook.


## 4. Spark compatibility smoke test

Run this after the runtime switch. The test verifies that Spark starts, DataFrame transformations work, SQL works, and Python-to-Spark execution succeeds.


In [ ]:
if 'spark' not in globals():
    raise RuntimeError('No active Spark session found. Attach this notebook to a Fabric Spark runtime first.')

from pyspark.sql import functions as F

numbers = spark.range(0, 10).withColumn('squared', F.col('id') * F.col('id'))
summary = numbers.agg(
    F.count('*').alias('row_count'),
    F.sum('squared').alias('sum_squared'),
).collect()[0]
sql_result = spark.sql('SELECT 2 + 2 AS result').collect()[0]['result']

assert summary['row_count'] == 10, summary
assert summary['sum_squared'] == 285, summary
assert sql_result == 4, sql_result

print('Spark compatibility smoke test passed on Spark', spark.version)


## 5. Optional Lakehouse read/write validation

If this notebook is attached to a Lakehouse, set `validation_path` to a temporary location and run the cell. Leave it as `None` to skip writes. The cell cleans up the temporary data when it finishes.


In [ ]:
validation_path = None  # Example: 'Files/tmp/runtime_upgrade_validation'

if validation_path:
    test_data = spark.createDataFrame([(1, 'runtime-2.0'), (2, 'validation')], ['id', 'label'])
    test_data.write.format('delta').mode('overwrite').save(validation_path)

    read_back = spark.read.format('delta').load(validation_path)
    assert read_back.count() == 2

    if 'mssparkutils' in globals():
        mssparkutils.fs.rm(validation_path, True)
    else:
        print(f'Validation succeeded. Remove temporary data manually from {validation_path}.')

    print(f'Lakehouse validation passed for {validation_path}')
else:
    print('Lakehouse validation skipped. Set validation_path to run it.')


## 6. Post-upgrade checklist

- Confirm the baseline output shows the expected Runtime 2.0 Spark and Python versions.
- Compare dependency versions with the Runtime 1.2 baseline and pin workspace/environment packages when needed.
- Rerun production notebooks or pipelines that depend on this runtime.
- Review any failures for changed Spark SQL behavior, removed legacy configuration, or package API changes.
- Keep a rollback path by retaining a copy of the original Runtime 1.2 notebook or environment settings until validation is complete.
